# Knowledge base (ChromaDB)

In [17]:
import numpy as np
import torch
import pandas as pd
import chromadb
from tqdm.notebook import tqdm
from PIL import Image
from transformers import CLIPModel, CLIPProcessor


CSV_PATH = "./test/augmented_clean_youtube_watch_log.csv"
THUMBNAIL_PATH = "./knowledge/thumbnails"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# model = model.to("mps" if torch.backends.mps.is_available() else "cpu")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="youtube_videos_test")

In [ ]:
import numpy as np
import torch
import pandas as pd
import chromadb
from tqdm.notebook import tqdm
from PIL import Image
from transformers import CLIPModel, CLIPProcessor


CSV_PATH = "./knowledge/clean_youtube_watch_log.csv"
THUMBNAIL_PATH = "./knowledge/thumbnails"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# model = model.to("mps" if torch.backends.mps.is_available() else "cpu")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="youtube_videos")

In [18]:
df = pd.read_csv(CSV_PATH)

def embed_text(text: str) -> np.ndarray:
    """Embed a text using CLIP"""
    inputs = proc(text=[text], return_tensors="pt", truncation=True, padding=True)  # WARNING! Truncation may lead to loss of information for long texts
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)  # Normalize
    return emb.to("cpu").numpy()


def embed_image(image: Image.Image) -> np.ndarray:
    """Embed an image using CLIP"""
    inputs = proc(images=image, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)  # Normalize
    return emb.to("cpu").numpy()

def search(query: str | Image.Image, user_id = 25, limit = 5) -> pd.DataFrame:
    """Search for similar videos by text or image."""
    query_vec = embed_image(query) if isinstance(query, Image.Image) else embed_text(query)

    similar_ids = collection.query(
        query_embeddings=query_vec,
        n_results=limit,
        where={"user_id": user_id}
    ).get("ids", [[]])[0]  

    return df[df["video_id"].isin(similar_ids)]

In [ ]:
search("The Vaccine Hoax They Don't Want You to Know")

ValueError: Cannot index with multidimensional key

## Find new videos (if any) to embed and add to the collection

In [19]:
ids_to_add = set(df["video_id"]) - set(collection.get(ids=None)["ids"])
print(f"Found {len(ids_to_add)} new videos to add to the collection.")

for video_id in tqdm(ids_to_add, desc="Building KB"):
    video_data = df[df["video_id"] == video_id].iloc[0]
    user_id = video_data["user_id"]
    title = video_data["video_title"]
    thumbnail = Image.open(f"{THUMBNAIL_PATH}/{video_id}.jpg")

    # Embed text and image, then fuse them (weighted average)
    text_emb = embed_text(title).squeeze()
    img_emb = embed_image(thumbnail).squeeze()
    fused_emb = 0.7 * text_emb + 0.3 * img_emb
    fused_emb = fused_emb / np.linalg.norm(fused_emb)  # Normalize the fused embedding

    # Add embedding to the collection
    collection.add(
        ids=[video_id],                     # ID
        embeddings=[fused_emb],             # Embedding (fused text + image)
        metadatas=[{"user_id": int(user_id)}]    # Metadata (user_id for filtering during retrieval)
    )

Found 1173 new videos to add to the collection.


Building KB:   0%|          | 0/1173 [00:00<?, ?it/s]

# Building the Agent (LangChain)

In [98]:
from dataclasses import dataclass
from typing import Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from PIL import Image
from typing import Literal
from typing import Optional
from langgraph.types import Command
from langchain.agents import AgentState
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain.messages import ToolMessage
from dataclasses import asdict
from langchain_groq import ChatGroq
from dataclasses import field
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
import uuid
from pathlib import Path
from typing import TypedDict

with open('llm-api-key.txt') as f:
    api_key = f.readline().strip()

# llm = ChatOllama(model="qwen3.5:2B", temperature=0)   # or qwen3:0.6b-q4_K_M
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    # model="llama-3.3-70b-versatile",
    api_key=api_key,
    temperature=0.4,
)
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     temperature=0,
#     api_key=api_key,
#     convert_system_message_to_human=True,
# )

## 1. Data types

In [99]:
@dataclass
class WatchItem:
    video_id: str
    video_title: str
    timestamp: str      # ISO format

    @property
    def thumbnail_url(self) -> str:
        return f'./knowledge/thumbnails/{self.video_id}.jpg'
    
    @classmethod
    def from_row(cls, row) -> "WatchItem":
        return cls(
            video_id=row['video_id'],
            video_title=row['video_title'],
            timestamp=row['watch_date']
        )

    def __repr__(self): # json
        d = asdict(self)
        d["thumbnail_url"] = self.thumbnail_url
        return str(d)

@dataclass
class BiasProfile:
    emotional_tone: Literal["low", "medium", "high"]
    sensationalism: Literal["low", "medium", "high"]
    topical_narrowing: Literal["low", "medium", "high"]
    echo_chamber_effect: Literal["low", "medium", "high"]
    dominant_topics: list[str]
    evidence_titles: list[str]
    overall_profile: str = field(
        metadata={"description": "Some comments from the model explaining why the scores were assigned."}
    )
    confidence: float = field(
        default=0.0,
        metadata={"description": "A confidence score (0-1) indicating how confident the model is in its bias assessment."}
    )

    def __repr__(self):
        return str(asdict(self))

## 2. Type definitions
> **⚠️ Models do NOT have access to these definitions, but tools do!**

In [100]:
# State (short-term memory)
class AgentState(AgentState):
    watch_history: list[WatchItem]                          # @dataclass
    bias_profile: Optional[BiasProfile]
    messages: Annotated[list[BaseMessage], add_messages]    # automatically append

# Context (static configuration)
class AgentContext(TypedDict):
    user_id: int

## 3. Tools

In [101]:
@tool
def retrieve_session(sort_asc: bool, limit: int, runtime: ToolRuntime[AgentContext, AgentState]) -> Command:
    """ Loads a user's watch session in the agent state.
    Args:
        sort_asc (bool): sort oldest first if True
        limit (int): maximum number of videos to retrieve
    Returns:
        Command: Updates the agent state with the retrieved watch history.
    """
    df = pd.read_csv(CSV_PATH)
    df = df[df['user_id'] == runtime.context["user_id"]]   # Filter by user ID from state
    df = df.sort_values("watch_date", ascending=sort_asc)
    df = df.head(limit) if limit else df
    result = df.apply(WatchItem.from_row, axis=1).tolist()  # Convert to WatchItem list

    return Command(
        update ={
            "watch_history": result,    # Update agent state with the retrieved watch history
            "messages": [ToolMessage(
                content=str(result),
                tool_call_id=runtime.tool_call_id
            )],
        },
    )


@tool
def analyze_bias_profile(runtime: ToolRuntime[AgentContext, AgentState]) -> Command:
    """ Analyzes a user's watch history and loads it in the agent state. 
    Requires a previous run of retrieve_session to have populated the watch_history in the state.
    Returns:
        Command: Updates the agent state with the bias profile.
    """
    watch_history = runtime.state["watch_history"]  # FIXME dict, not object?

    if not watch_history:
        return Command(
            update = {
                "messages": [ToolMessage(
                    content="No watch history found. Please run retrieve_session first.",
                    tool_call_id=runtime.tool_call_id
                )],
            },
        )
    
    global bias_analyzer_agent  
    prompt = HumanMessage(content="Analyze this watch history for bias: " + str(watch_history))
    raw = bias_analyzer_agent.invoke({ "messages": [prompt] })  # type: ignore
    result: BiasProfile = raw["structured_response"]    # FIXME dict, not object?

    return Command(
        update = { 
            "bias_profile": result,
            "messages": [ToolMessage(
                content=asdict(result), # bug! can't use str(...) as would use default __rept__
                tool_call_id=runtime.tool_call_id
            )],
        },
    )


@tool
def find_similar_videos(title_or_thumbnail_url: str, limit: int, runtime: ToolRuntime[AgentContext, AgentState]) -> list[WatchItem]:
    """Invokes the VectorDB to find similar videos by title and/or thumbnail.
    Args:
        title_or_thumbnail_url (str): A title or thumbnail (local file path) to find similar videos for.
        limit (int): The number of similar videos to return. Defaults to 5.
    Returns:
        list[WatchItem]: A list of similar videos
    """
    # Embed the input title and thumbnail

    # TODO cross-user search?
    # TODO if title or thumbnail exists, augment query with the corresponding one to improve results
    # TODO check if file exists and manage URL-based thumbnails
    if title_or_thumbnail_url.lower().endswith((".jpg", ".jpeg", ".png")):
        # treat as thumbnail path
        query_vec = embed_image(Image.open(title_or_thumbnail_url))
    else:
        # treat as text
        query_vec = embed_text(title_or_thumbnail_url)

    # Query the vector database for similar videos
    similar_ids = collection.query(
        query_embeddings=query_vec,
        n_results=limit or 5,
        where={"user_id": runtime.context["user_id"]}
    ).get("ids", [[]])[0]  

    return (
        df[df["video_id"].isin(similar_ids)]
        .apply(WatchItem.from_row, axis=1)
        .tolist()
    )


TOOLS = [retrieve_session, analyze_bias_profile, find_similar_videos]

## 4. Agents

In [102]:
# ---------- Node 1: General-Purpose LLM agent  ----------
SYSTEM_PROMPT = """
You are a YouTube watch-history analysis agent focused on bias, polarization, sensationalism, clickbait, emotional tone, and rabbit-hole effects.

# Rules:
- If you are sure about the user’s intent AND it matches a tool’s purpose, call that tool with appropriately typed arguments. Otherwise, do not call any tool and ask for clarification.
- When asked about bias, polarization, sensationalism, clickbait, emotional tone, or rabbit-hole effects:
  - First ensure the watch history was retrieved using the relative tool.
  - Then call the analyze_bias_profile tool to obtain a structured bias profile.
- Base all claims on factual data from watch history and tool results. For each claim, mention a specific example from the watch history that supports it (e.g., a video title or topic). Avoid vague generalizations.
- Be cautious and do not overclaim ideology; if unsure, say that the evidence is weak or ambiguous and optionally ask for clarification.
- When given an image, use the find_similar_videos tool immediately, passing the image's local file path directly to the tool.
- When asked to display an image, respond with the [markdown tag](URL), where URL is the best image URL from user input, state, memory, or tools.
- Do not call tools multiple times in the same turn; if you need to call another tool, wait for the next turn to do so.
- Never use tables.
"""
agent = create_agent(
    llm,
    tools=TOOLS,
    state_schema=AgentState,
    context_schema=AgentContext,
    system_prompt=SYSTEM_PROMPT,
    middleware=[ToolCallLimitMiddleware(run_limit=1)],   # Only one tool call per turn
    checkpointer=InMemorySaver(serde=JsonPlusSerializer(
        allowed_msgpack_modules=[
            ("__main__", "WatchItem"),
            ("__main__", "BiasProfile"),
        ]
    )),
)


# ---------- Node 2: Bias Analysis sub-agent  ----------
BIAS_AGENT_PROMPT = """
You analyze a single YouTube watch session and return a structured bias profile and assessment of rabbit-hole effects.

# Rules
- Use only the provided watch-session records; do not assume any additional history.
- If no watch history is provided, explicitly refuse to analyze.
- Be cautious and conservative in your conclusions.
- Do not overclaim ideology; avoid assigning strong labels without clear evidence.
- Focus your analysis on:
  - Ideological or political lean (if any).
  - Topic concentration versus diversity.
  - Emotional tone and sensationalism.
  - Escalation or narrowing over time (rabbit-hole behavior).
"""
bias_analyzer_agent = create_agent(
    llm,
    state_schema=AgentState,
    system_prompt=BIAS_AGENT_PROMPT,
    response_format=BiasProfile,    # TODO check ToolStrategy and ProviderStrategy
)

# Serve
Open the browser at [index.html](http://127.0.0.1:8000)

In [103]:
import markdown

def md_to_html(md_source: str) -> str:
    """Converts markdown to HTML and sanitizes image paths"""
    html = markdown.markdown(md_source, extensions=['tables', 'fenced_code'])
    # Replace image src paths
    html = html.replace('src="./knowledge/thumbnails/', 'src="/thumbnails/')
    html = html.replace('src="./temp/', 'src="/uploads/')
    return html

In [ ]:
from fastapi import FastAPI, Form, File, UploadFile, HTTPException
from tempfile import NamedTemporaryFile
import uvicorn
from fastapi.staticfiles import StaticFiles
import shutil
import logging

log = logging.getLogger(f"uvicorn.{__name__}")
app = FastAPI()

@app.get("/api/users")
def get_available_users() -> list[int]:
    return pd.read_csv(CSV_PATH)["user_id"].unique().tolist()

@app.post("/api/chat")
async def chat(
    user_id: int = Form(...),   # multipart/form-data
    message: str = Form(...),
    thread_id: str | None = Form(None),
    image: UploadFile | None = File(None),
) -> dict:
    if not message.strip():
        raise HTTPException(400, "message cannot be empty")

    # Generate thread ID if not provided
    thread_id = thread_id or uuid.uuid4().hex

    # Single optional image as multipart file upload
    if image:
        tmp = NamedTemporaryFile(dir="./temp", delete=False, suffix=image.filename, prefix="")
        shutil.copyfileobj(image.file, tmp)
        message += "\nAttached image: " + tmp.name

    # Invoke model
    messages = agent.invoke(
        {"messages": [HumanMessage(content=message)]},
        config={"configurable": {"thread_id": thread_id}},
        context={"user_id": user_id},
    ).get("messages", [])

    html_response = ""
    for msg in messages:
        log.info(f"User[{user_id}] - Thread[{thread_id}] {msg.type}: {msg.content}")
        # Obtain HTML from last message
        html_response = md_to_html(str(msg.content))

    # Stream model response
    return {"thread_id": thread_id, "html_response": html_response}

# Serve the static content
app.mount("/thumbnails", StaticFiles(directory="knowledge/thumbnails"))
app.mount("/uploads", StaticFiles(directory="temp"))
app.mount("/", StaticFiles(directory="frontend", html=True))

# Start the server
await uvicorn.Server(uvicorn.Config(app)).serve()

INFO:     Started server process [69071]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:49274 - "GET / HTTP/1.1" 304 Not Modified
INFO:     127.0.0.1:49274 - "GET /.well-known/appspecific/com.chrome.devtools.json HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49274 - "GET /script.js HTTP/1.1" 304 Not Modified
INFO:     127.0.0.1:49274 - "GET /api/users HTTP/1.1" 200 OK


INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] human: look at this image
Attached image: /Users/francesco/Documents/dev/INFS4205-Personalised-Multimodal-Agent-System/temp/gkjejaj_Sef49e364e982431bb7db80199b4fef4bu.jpg.webp
INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] ai: 
INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] tool: [{'video_id': 'EKeF85hap84', 'video_title': '2018 06 07 16 50 13 2', 'timestamp': '2018-08-01 22:04:31', 'thumbnail_url': './knowledge/thumbnails/EKeF85hap84.jpg'}, {'video_id': 'qcJ1Ww0m9aA', 'video_title': '16_3 (2)', 'timestamp': '2018-06-08 22:18:45', 'thumbnail_url': './knowledge/thumbnails/qcJ1Ww0m9aA.jpg'}, {'video_id': 'WloR27rFcAA', 'video_title': 'I tech solution', 'timestamp': '2018-06-08 21:58:06', 'thumbnail_url': './knowledge/thumbnails/WloR27rFcAA.jpg'}, {'video_id': 'CKRjXZSgg-g', 'video_title': 'Data Scraping and Mining From Websites Free No Coding Required', 'timestamp': '2018-03-22 04:08:26', 'thumbna

INFO:     127.0.0.1:49275 - "POST /api/chat HTTP/1.1" 200 OK


INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] human: look at this image
Attached image: /Users/francesco/Documents/dev/INFS4205-Personalised-Multimodal-Agent-System/temp/gkjejaj_Sef49e364e982431bb7db80199b4fef4bu.jpg.webp
INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] ai: 
INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] tool: [{'video_id': 'EKeF85hap84', 'video_title': '2018 06 07 16 50 13 2', 'timestamp': '2018-08-01 22:04:31', 'thumbnail_url': './knowledge/thumbnails/EKeF85hap84.jpg'}, {'video_id': 'qcJ1Ww0m9aA', 'video_title': '16_3 (2)', 'timestamp': '2018-06-08 22:18:45', 'thumbnail_url': './knowledge/thumbnails/qcJ1Ww0m9aA.jpg'}, {'video_id': 'WloR27rFcAA', 'video_title': 'I tech solution', 'timestamp': '2018-06-08 21:58:06', 'thumbnail_url': './knowledge/thumbnails/WloR27rFcAA.jpg'}, {'video_id': 'CKRjXZSgg-g', 'video_title': 'Data Scraping and Mining From Websites Free No Coding Required', 'timestamp': '2018-03-22 04:08:26', 'thumbna

...

INFO:     127.0.0.1:49468 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49468 - "GET /thumbnails/zpsLeylY5-Y.jpg HTTP/1.1" 200 OK


INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] human: look at this image
Attached image: /Users/francesco/Documents/dev/INFS4205-Personalised-Multimodal-Agent-System/temp/gkjejaj_Sef49e364e982431bb7db80199b4fef4bu.jpg.webp
INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] ai: 
INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] tool: [{'video_id': 'EKeF85hap84', 'video_title': '2018 06 07 16 50 13 2', 'timestamp': '2018-08-01 22:04:31', 'thumbnail_url': './knowledge/thumbnails/EKeF85hap84.jpg'}, {'video_id': 'qcJ1Ww0m9aA', 'video_title': '16_3 (2)', 'timestamp': '2018-06-08 22:18:45', 'thumbnail_url': './knowledge/thumbnails/qcJ1Ww0m9aA.jpg'}, {'video_id': 'WloR27rFcAA', 'video_title': 'I tech solution', 'timestamp': '2018-06-08 21:58:06', 'thumbnail_url': './knowledge/thumbnails/WloR27rFcAA.jpg'}, {'video_id': 'CKRjXZSgg-g', 'video_title': 'Data Scraping and Mining From Websites Free No Coding Required', 'timestamp': '2018-03-22 04:08:26', 'thumbna

INFO:     127.0.0.1:49689 - "POST /api/chat HTTP/1.1" 200 OK


INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] human: look at this image
Attached image: /Users/francesco/Documents/dev/INFS4205-Personalised-Multimodal-Agent-System/temp/gkjejaj_Sef49e364e982431bb7db80199b4fef4bu.jpg.webp
INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] ai: 
INFO:     User[25] - Thread[3be0bbf851df41c8918b7eb6578c0cc6] tool: [{'video_id': 'EKeF85hap84', 'video_title': '2018 06 07 16 50 13 2', 'timestamp': '2018-08-01 22:04:31', 'thumbnail_url': './knowledge/thumbnails/EKeF85hap84.jpg'}, {'video_id': 'qcJ1Ww0m9aA', 'video_title': '16_3 (2)', 'timestamp': '2018-06-08 22:18:45', 'thumbnail_url': './knowledge/thumbnails/qcJ1Ww0m9aA.jpg'}, {'video_id': 'WloR27rFcAA', 'video_title': 'I tech solution', 'timestamp': '2018-06-08 21:58:06', 'thumbnail_url': './knowledge/thumbnails/WloR27rFcAA.jpg'}, {'video_id': 'CKRjXZSgg-g', 'video_title': 'Data Scraping and Mining From Websites Free No Coding Required', 'timestamp': '2018-03-22 04:08:26', 'thumbna

INFO:     127.0.0.1:49773 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49773 - "GET /thumbnails/btmnWqh5IEk.jpg HTTP/1.1" 200 OK


# Testing
> **⚠️ WARNING: Don't forget to run cell above!**

## Interactive

In [ ]:
user_id = 25
thread_id = uuid.uuid4().hex

print("Agent 🤖 is ready. Type 'exit' to quit.\n")
while True:
    user = HumanMessage(content=input("\nYou: ").strip())
    if user.content == '' or user.content.lower() == "exit":
        break
    
    user.pretty_print()
    for step in agent.stream(
        { "messages" : [user] },
        config={ "configurable": {"thread_id": thread_id} },    # Add memory
        context={"user_id": user_id},                           # Retrieved in tools
    ):
        for update in step.values():
            for m in update.get("messages", []) if update else []:
                m.pretty_print()

## Debugging

In [ ]:
user_id = 25
thread_id = uuid.uuid4().hex

dummy_runtime = ToolRuntime(
    context=AgentContext(user_id=user_id),
    state=AgentState(
        # watch_history=[WatchItem.from_row(df[df["user_id"] == user_id].iloc[0])],  # Example state with one video
        watch_history=[WatchItem("F8wJdvXK5yU", "Far right for the win!!", "2023-01-01T12:00:00Z")],  # Example state with one video
        bias_profile=None,
        messages=[]
    ),
    tool_call_id="test_call_1",
    config={ "configurable": {"thread_id": thread_id} },
    stream_writer=lambda x: None,
    store=lambda x: None
)

# command = retrieve_session.func(sort_asc=True, limit=5, runtime=dummy_runtime)
command = analyze_bias_profile.func(runtime=dummy_runtime) # type: ignore
command.update["messages"][0].pretty_print()